In [1]:
%matplotlib inline

%load_ext autoreload
%autoreload 2

In [2]:
import sys

sys.path.append('../..')

In [3]:
import numpy as np
import torch

from stable_baselines3 import PPO, DQN
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.vec_env import DummyVecEnv, SubprocVecEnv
from stable_baselines3.common.utils import set_random_seed
from stable_baselines3.common.monitor import Monitor

In [4]:
from vimms.Common import POSITIVE
from vimms.ChemicalSamplers import UniformRTAndIntensitySampler, GaussianChromatogramSampler, UniformMZFormulaSampler

from vimms_gym.env import DDAEnv

/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/statsmodels/compat/pandas.py:61: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import Int64Index as NumericIndex
/opt/anaconda3/envs/vimms-gym/lib/python3.9/site-packages/psims/mzmlb/writer.py:15: UserWarning: hdf5plugin is missing! Only the slower GZIP compression scheme will be available! Please install hdf5plugin to be able to use Blosc.
  warnings.warn(


# 1. Parameters

In [5]:
# n_chemicals = (200, 500)
# mz_range = (100, 600)
# rt_range = (0, 300)
# intensity_range = (1E5, 1E10)

In [6]:
n_chemicals = (2000, 5000)
mz_range = (100, 600)
rt_range = (200, 1000)
intensity_range = (1E4, 1E10)

In [7]:
min_mz = mz_range[0]
max_mz = mz_range[1]
min_rt = rt_range[0]
max_rt = rt_range[1]
min_log_intensity = np.log(intensity_range[0])
max_log_intensity = np.log(intensity_range[1])

In [8]:
isolation_window = 0.7
N = 10
rt_tol = 120
mz_tol = 10
min_ms1_intensity = 5000
ionisation_mode = POSITIVE

enable_spike_noise = True
noise_density = 0.1
noise_max_val = 1E3

In [9]:
mz_sampler = UniformMZFormulaSampler(min_mz=min_mz, max_mz=max_mz)
ri_sampler = UniformRTAndIntensitySampler(min_rt=min_rt, max_rt=max_rt,
                                          min_log_intensity=min_log_intensity,
                                          max_log_intensity=max_log_intensity)
cr_sampler = GaussianChromatogramSampler()
samplers = {
    'mz': mz_sampler,
    'rt_intensity': ri_sampler,
    'chromatogram': cr_sampler
}

In [10]:
params = {
    'chemical_creator': {
        'mz_range': mz_range,
        'rt_range': rt_range,
        'intensity_range': intensity_range,
        'n_chemicals': n_chemicals,
        'mz_sampler': mz_sampler,
        'ri_sampler': ri_sampler,
        'cr_sampler': cr_sampler,
    },
    'noise': {
        'enable_spike_noise': enable_spike_noise,
        'noise_density': noise_density,
        'noise_max_val': noise_max_val,
        'mz_range': mz_range
    },
    'env': {
        'ionisation_mode': ionisation_mode,
        'rt_range': rt_range,
        'isolation_window': isolation_window,
        'mz_tol': mz_tol,
        'rt_tol': rt_tol,
    }
}

In [11]:
max_peaks = 200
in_dir = 'results'

In [12]:
deterministic = True
cpu_limit = 40

# 2. Train PPO

In [13]:
def make_env(rank, seed=0):
    def _init():
        env = DDAEnv(max_peaks, params)
        env.seed(rank)
        env = Monitor(env)        
        return env
    set_random_seed(seed)
    return _init

### Using DDAEnv

In [14]:
env = DDAEnv(max_peaks, params)
check_env(env)

In [15]:
env_name = 'DDAEnv'

### Parameter set 1

In [ ]:
# default parameters
# learning_rate = 0.0003
# batch_size = 64
# n_steps = 2048
# ent_coef = 0.0
# gamma = 0.99
# gae_lambda = 0.95

In [16]:
# parameter set 1
learning_rate = 0.0003
batch_size = 512
n_steps = 2048
ent_coef = 0.001
gamma = 0.90
gae_lambda = 0.90
hidden_nodes = 512
total_timesteps = 10E6

net_arch = [dict(pi=[hidden_nodes, hidden_nodes], vf=[hidden_nodes, hidden_nodes])]
policy_kwargs = dict(net_arch=net_arch)

In [17]:
model_name = 'PPO'
fname = '%s/%s_%s.zip' % (in_dir, env_name, model_name)

In [18]:
fname

'results/CoverageEnv_PPO_1.zip'

In [ ]:
num_cpu = int(cpu_limit / 2)
torch.set_num_threads(num_cpu)
env = SubprocVecEnv([make_env(i) for i in range(num_cpu)])

tensorboard_log = './%s/%s_%s_tensorboard' % (in_dir, env_name, model_name)

model = PPO('MultiInputPolicy', env, 
            tensorboard_log=tensorboard_log, 
            learning_rate=learning_rate, batch_size=batch_size, n_steps=n_steps, 
            ent_coef=ent_coef, gamma=gamma, gae_lambda=gae_lambda, policy_kwargs=policy_kwargs)
model.learn(total_timesteps=total_timesteps)
model.save(fname)

# 3. Train DQN

In [ ]:
# model_name = 'DQN'
# fname = '%s/%s_%s.zip' % (in_dir, env_name, model_name)

In [ ]:
# # original parameters
# learning_rate = 0.0001
# batch_size = 32
# gamma = 0.99
# exploration_fraction = 0.1
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.05
# hidden_nodes = 64
# total_timesteps = 5E6

# net_arch = [hidden_nodes, hidden_nodes]
# policy_kwargs = dict(net_arch=net_arch)

In [ ]:
# # modified parameters
# learning_rate = 0.0001
# batch_size = 512
# gamma = 0.90
# exploration_fraction = 0.25
# exploration_initial_eps = 1.0
# exploration_final_eps = 0.10
# hidden_nodes = 512
# total_timesteps = 5E6

# net_arch = [hidden_nodes, hidden_nodes]
# policy_kwargs = dict(net_arch=net_arch)

In [ ]:
# num_cpu = cpu_limit
# torch.set_num_threads(num_cpu)
# env = DDAEnv(max_peaks, params)
# env = Monitor(env)
# env = DummyVecEnv([lambda: env])

# tensorboard_log = './%s/%s_%s_tensorboard' % (in_dir, env_name, model_name)

# model = DQN('MultiInputPolicy', env, 
#             tensorboard_log=tensorboard_log, 
#             learning_rate=learning_rate, batch_size=batch_size, gamma=gamma,
#             exploration_fraction=exploration_fraction, exploration_initial_eps=exploration_initial_eps, exploration_final_eps=exploration_final_eps, 
#             policy_kwargs=policy_kwargs)
# model.learn(total_timesteps=total_timesteps)
# model.save(fname)